In [ ]:
!nvidia-smi

#1.&nbsp;Thu thập và dán nhãn hình ảnh đào tạo




- Thư mục `images` chứa các hình ảnh
- Thư mục `labels` chứa các nhãn ở định dạng chú thích YOLO
- Tệp bản đồ nhãn `classes.txt` chứa tất cả các lớp
- Tệp `notes.json` chứa thông tin dành riêng cho Label Studio (có thể bỏ qua tệp này)

<p align=center>
<img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/zipped-data-example.png" height=""><br>
<i> <a href="https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/candy_data_06JAN25.zip"

# 2.&nbsp;Tải lên bộ dữ liệu hình ảnh và chuẩn bị dữ liệu huấn luyện

## 2.1 Upload images
Trước tiên, chúng ta cần tải bộ dữ liệu lên Colab. Dưới đây là một vài tùy chọn để di chuyển thư mục `data.zip` vào phiên bản Colab này.


Tải tệp `data.zip` lên phiên bản Google Colab bằng cách nhấp vào biểu tượng "Tệp" ở phía bên trái trình duyệt, sau đó nhấp vào biểu tượng "Tải lên bộ nhớ phiên". Chọn thư mục zip để tải lên.

<p>
<br>
<img src="https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/doc/upload-colab-files.png" height="240">
</p>


## 2.2 Chia ảnh thành các thư mục huấn luyện và kiểm tra ( giải nén)


In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/data.zip -d /content/custom_data

Ultralytics yêu cầu cấu trúc thư mục cụ thể để lưu trữ dữ liệu huấn luyện cho các mô hình. Thư mục gốc được đặt tên là “data”. Bên trong, có hai thư mục chính:

* **Train**: Đây là những hình ảnh thực tế được sử dụng để huấn luyện mô hình. Trong một epoch huấn luyện, mọi hình ảnh trong tập huấn luyện đều được đưa vào mạng nơ-ron. Thuật toán huấn luyện điều chỉnh trọng số của mạng để phù hợp với dữ liệu trong hình ảnh.

* **Validation**: Những hình ảnh này được sử dụng để kiểm tra hiệu suất của mô hình vào cuối mỗi epoch huấn luyện.

Trong mỗi thư mục này có một thư mục “images” và một thư mục “labels”, lần lượt chứa các tệp hình ảnh và tệp chú thích.

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

--2026-01-13 11:32:58--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-01-13 11:32:58 (57.9 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/train/images.
Created folder at /content/data/train/labels.
Created folder at /content/data/validation/images.
Created folder at /content/data/validation/labels.
Number of image files: 50
Number of annotation files: 50
Images moving to train: 45
Images moving to validation: 5


# 3.&nbsp;Cài đặt thư viện (Ultralytics)
Tiếp theo, chúng ta sẽ cài đặt thư viện Ultralytics trong phiên bản Google Colab này. Thư viện Python này sẽ được sử dụng để huấn luyện mô hình YOLO.

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.6 MB/s eta 0:00:00


# 4.&nbsp;Configure Training


Đây là bước cuối cùng trước khi chúng ta có thể chạy huấn luyện: chúng ta cần tạo tệp cấu hình YAML huấn luyện Ultralytics. Tệp này chỉ định vị trí dữ liệu huấn luyện và xác thực, đồng thời định nghĩa các lớp của mô hình.

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 24
names:
- But
- Keo
- Thuoc
- bang dinh
- bat
- bat lua
- bitcoin
- bua
- but xoa
- dao doc giay
- dau an
- dieu khien
- dua
- ghim
- giay
- kim
- luoc
- may anh
- my pham
- quat tay
- sach
- the
- thia
- usb


# 5.&nbsp;Train Model

## 5.1 Training Parameters

**Số lượng epoch (`epochs`)**

Trong học máy, một “epoch” là một lần chạy qua toàn bộ tập dữ liệu huấn luyện. Việc thiết lập số lượng epoch sẽ quyết định thời gian huấn luyện của mô hình. Số lượng epoch tốt nhất phụ thuộc vào kích thước của tập dữ liệu và kiến ​​trúc của mô hình. Nếu tập dữ liệu của bạn có ít hơn 200 hình ảnh, điểm khởi đầu tốt là 60 epoch. Nếu tập dữ liệu của bạn có hơn 200 hình ảnh, điểm khởi đầu tốt là 40 epoch.

**Độ phân giải (`imgsz`)**

Độ phân giải có tác động lớn đến tốc độ và độ chính xác của mô hình: mô hình có độ phân giải thấp hơn sẽ có tốc độ nhanh hơn nhưng độ chính xác thấp hơn. Các mô hình YOLO thường được huấn luyện và suy luận ở độ phân giải 640x640. Tuy nhiên, nếu muốn mô hình của mình chạy nhanh hơn hoặc biết rằng bạn sẽ làm việc với hình ảnh có độ phân giải thấp, chúng ta quyết dịnh thử sử dụng độ phân giải thấp hơn như 480x480.


## 5.2 Training

Chạy đoạn mã sau để bắt đầu huấn luyện:

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj

Thuật toán huấn luyện sẽ phân tích các hình ảnh trong thư mục huấn luyện và xác thực, sau đó bắt đầu huấn luyện mô hình. Vào cuối mỗi epoch huấn luyện, chương trình sẽ chạy mô hình trên tập dữ liệu xác thực và báo cáo kết quả mAP, độ chính xác và độ thu hồi. Khi quá trình huấn luyện tiếp tục, mAP thường sẽ tăng lên sau mỗi epoch. Quá trình huấn luyện sẽ kết thúc khi hoàn thành số epoch được chỉ định bởi tham số `epochs`.

#6.Test
Mô hình đã được huấn luyện.Các lệnh bên dưới sẽ chạy mô hình trên các hình ảnh trong thư mục xác thực và sau đó hiển thị kết quả cho 10 hình ảnh đầu tiên.

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,422,088 parameters, 0 gradients, 21.4 GFLOPs

image 1/5 /content/data/validation/images/1dfdf95d-z7370826053581_6f06ab26f7d1e5d105e0d5d0c1b888a3.jpg: 640x480 1 my pham, 52.9ms
image 2/5 /content/data/validation/images/79df62e6-z7370826226120_526a79af5654a7b918d4cff406d458b8.jpg: 640x480 1 But, 1 bitcoin, 1 my pham, 12.4ms
image 3/5 /content/data/validation/images/c4a48843-z7370826357792_91e1f43417f3233d1ca474af8b0ba895.jpg: 480x640 (no detections), 56.4ms
image 4/5 /content/data/validation/images/d4d72214-z7370826026015_06c312b893d6b84abe3c4e03267b4f54.jpg: 640x480 (no detections), 13.2ms
image 5/5 /content/data/validation/images/e189a915-z7370826143194_ad4fa5a0f346bee544278c2ef26d146c.jpg: 480x640 1 bang dinh, 1 my pham, 1 quat tay, 13.2ms
Speed: 2.7ms preprocess, 29.6ms inference, 4.3ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /content/runs

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')


#7.&nbsp;Deploy Model

## Download YOLO Model


Đầu tiên, nén và tải xuống mô hình đã được huấn luyện bằng cách chạy các khối mã bên dưới.

In [ ]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

/content/my_model
  adding: my_model.pt (deflated 8%)
  adding: train/ (stored 0%)
  adding: train/val_batch0_pred.jpg (deflated 13%)
  adding: train/BoxP_curve.png (deflated 11%)
  adding: train/train_batch1.jpg (deflated 3%)
  adding: train/train_batch0.jpg (deflated 3%)
  adding: train/results.csv (deflated 63%)
  adding: train/train_batch2.jpg (deflated 4%)
  adding: train/BoxPR_curve.png (deflated 17%)
  adding: train/BoxF1_curve.png (deflated 10%)
  adding: train/train_batch151.jpg (deflated 8%)
  adding: train/args.yaml (deflated 53%)
  adding: train/BoxR_curve.png (deflated 13%)
  adding: train/weights/ (stored 0%)
  adding: train/weights/best.pt (deflated 8%)
  adding: train/weights/last.pt (deflated 8%)
  adding: train/train_batch152.jpg (deflated 5%)
  adding: train/confusion_matrix.png (deflated 22%)
  adding: train/confusion_matrix_normalized.png (deflated 21%)
  adding: train/labels.jpg (deflated 30%)
  adding: train/val_batch0_labels.jpg (deflated 13%)
  adding: train/tr

In [ ]:
# This takes forever for some reason, you can also just download the model from the sidebar
from google.colab import files

files.download('/content/my_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>